In [1]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import joblib

In [2]:
def get_file(path):
    """
    recupère le fichier csv donnée avec le séparateur ';' et retourne un dataFrame.

    Args:
        path (str): Le chemin vers le fichier CSV.

    Returns:
        pd.DataFrame: Les données chargées.
    """

    # ouvir le fichier en dataframe
    df = pd.read_csv(path, sep=",")
    return df

In [3]:
def cal_fiabilite(valeurs_attendues, valeurs_predites, noms_colonnes):
    """
    Calcule le taux de rapprochement entre valeurs attendues et prédites.
    """
    # mettre les valeurs dans une array
    valeurs_attendues = np.array(valeurs_attendues)
    valeurs_predites = np.array(valeurs_predites)

    scores = {}

    for i, col in enumerate(noms_colonnes):
        # mettre toutes les colones une par une variable
        attendues = valeurs_attendues[:, i]
        predites = valeurs_predites[:, i]

        # le mask pour ne pas faire les division par 0
        mask = attendues != 0

        # calul du taux de diffrence
        taux_erreur = np.mean(
            np.abs(attendues[mask] - predites[mask]) / np.abs(attendues[mask])
        )

        # inverser le taux pour avoir le taux de rapprochement
        taux_rapprochement = max(0, (1 - taux_erreur) * 100)

        # stocker la valeurs dans une variable
        scores[col] = round(taux_rapprochement, 2)

    #stocker la moyenne des scores
    scores["Global"] = round(np.mean(list(scores.values())), 2)

    return scores

In [4]:

# load 
df = get_file("cleaned_dataset.csv")
#les colonnes d'entre du model
input_col = [
    "Annee",
    "Mois",
    "Service",
    "Gare de départ",
    "Gare d'arrivée",
    "Durée moyenne du trajet",
    "Nombre de circulations prévues",
]
# les colonnes de sortie du model
output_col = [  # "Nombre de trains annulés",
    "Nombre de trains en retard au départ",
    "Retard moyen des trains en retard au départ",
    "Nombre de trains en retard à l'arrivée",
    "Retard moyen des trains en retard à l'arrivée",
    "Retard moyen de tous les trains à l'arrivée",
    "Nombre trains en retard > 15min",
    #"Retard moyen trains en retard > 15 (si liaison concurrencée par vol)",
    "Nombre trains en retard > 30min",
    "Nombre trains en retard > 60min",
    # "Prct retard pour causes externes",
    # "Prct retard pour cause infrastructure",
    # "Prct retard pour cause gestion trafic",
    # "Prct retard pour cause matériel roulant",
    # "Prct retard pour cause gestion en gare et réutilisation de matériel",
    # "Prct retard pour cause prise en compte voyageurs (affluence, gestions PSH, correspondances)"
]
label_encoders = {}

# changer les string en int avec un label 
for col in ["Service", "Gare de départ", "Gare d'arrivée"]:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

# exporter le label 
joblib.dump(label_encoders, "label_encoders.joblib")

#stocker toutes les colonnes de input et de output separement 
inputt = df[input_col]
output = df[output_col]

# separer les output et input pour les test et les entrainement 
input_train, input_test, output_train, output_test = train_test_split(
    inputt, output, test_size=0.1
)

# crée le modele de forest avec 50 arbre 
modele_rf = RandomForestRegressor(n_estimators=50, n_jobs=-1)
# entrainer le modele
modele_rf.fit(input_train, output_train)
#stocker les prediction faites 
prediction_rf = modele_rf.predict(input_test)

#crée le modele tree
model = DecisionTreeRegressor()
model.fit(input_train, output_train)
prediction = model.predict(input_test)

#stocker les score de chaque arbre
scores_rf = cal_fiabilite(output_test, prediction_rf, output_col)
scores = cal_fiabilite(output_test, prediction, output_col)

# calculer la moyenne des délais sur les données d'entrainement
mean_delays = output_train.mean()

# crée une prediction de base en répétant la moyenne pour chaque ligne de test
base_prediction = np.tile(mean_delays.values, (len(output_test), 1))

# calcule le score de la moyenne
scores_base = cal_fiabilite(output_test, base_prediction, output_col)

print(f"Score Baseline : {scores_base['Global']}")
print(f"Score Forest : {scores_rf['Global']} ")
print(f"Score Tree : {scores['Global']} ")

#exporter les modeléles
joblib.dump(modele_rf, "model_rf.joblib")
joblib.dump(model, "model.joblib")

Score Baseline : 8.36
Score Forest : 46.61 
Score Tree : 32.18 


['model.joblib']